In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

from ml.common import CAP, SEED, load_fd, select_sensors
from ml.sequences import make_windows
from ml.models import GRUReg

torch.manual_seed(SEED)
np.random.seed(SEED)
print("torch", torch.__version__)

WIN = 30
train, test, rul = load_fd("FD001")
keep = select_sensors(train)

# Neural networks need inputs on a similar scale: standardise with TRAIN statistics only
mean, std = train[keep].mean(), train[keep].std()
train_s, test_s = train.copy(), test.copy()
train_s[keep] = (train[keep] - mean) / std
test_s[keep] = (test[keep] - mean) / std
print(len(keep), "sensors")

torch 2.15.0+cpu
14 sensors


In [2]:
X_train, u_train = make_windows(train_s, keep, WIN)
y_train = train_s["RUL"].to_numpy(dtype=np.float32)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("NaNs:", int(np.isnan(X_train).sum()))

X_train: (20631, 30, 14) | y_train: (20631,)
NaNs: 0


In [3]:
eng = train_s[train_s["unit"] == 1].copy()
w_a, _ = make_windows(eng, keep, WIN)

future_wiped = eng.copy()
future_wiped.loc[future_wiped["cycle"] > 50, keep] = 0.0   # destroy everything after cycle 50
w_b, _ = make_windows(future_wiped, keep, WIN)

print("windows for cycles 1-50 unchanged:", np.allclose(w_a[:50], w_b[:50]))
print("window for cycle 51 changed:", not np.allclose(w_a[50], w_b[50]))

windows for cycles 1-50 unchanged: True
window for cycle 51 changed: True


In [4]:
X_test_all, u_test = make_windows(test_s, keep, WIN)
last_idx = test_s.reset_index(drop=True).groupby("unit").tail(1).index.to_numpy()
X_test = X_test_all[last_idx]
y_test = np.minimum(rul["RUL"].values, CAP).astype(np.float32)

print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("first engine ids:", u_test[last_idx][:5])
print("all test engines have >= 30 cycles (no padding in final windows):",
      bool((test.groupby("unit")["cycle"].max() >= WIN).all()))

X_test: (100, 30, 14) | y_test: (100,)
first engine ids: [1 2 3 4 5]
all test engines have >= 30 cycles (no padding in final windows): True


In [5]:
ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
loader = DataLoader(ds, batch_size=256, shuffle=True)
xb, yb = next(iter(loader))
print("batch:", tuple(xb.shape), tuple(yb.shape))

model = GRUReg(n_features=len(keep), hidden=64)
print(sum(p.numel() for p in model.parameters()), "trainable parameters")

with torch.no_grad():
    out = model(xb)
print("output:", tuple(out.shape))

batch: (256, 30, 14) (256,)
15425 trainable parameters
output: (256,)
